In [12]:
# Cell 1: Import necessary libraries
import numpy as np
import os
import time
from pydrake.all import (
    DiagramBuilder, AddMultibodyPlantSceneGraph, Parser, RigidTransform, RotationMatrix,
    Role, MeshcatVisualizer, StartMeshcat, RationalForwardKinematics, CspaceFreePolytope,
    SeparatingPlaneOrder, Rgba, InverseKinematics, RollPitchYaw,
    LinearEqualityConstraint, Sphere, Parallelism, AddDefaultVisualization, 
    ConnectPlanarSceneGraphVisualizer, IrisFromCliqueCoverOptions, 
    IrisInConfigurationSpaceFromCliqueCover, RandomGenerator, RobotDiagramBuilder, 
    SceneGraphCollisionChecker, MultibodyPlant, SceneGraph, 
    SolverOptions, CommonSolverOption, GeometrySet, ScsSolver
)
from pydrake.geometry.optimization import GraphOfConvexSetsOptions, HPolyhedron, VPolytope, Point, Hyperellipsoid
from pydrake.geometry.optimization import ConvexHull as DrakeConvexHull
from pydrake.planning import GcsTrajectoryOptimization
from pydrake.solvers import MathematicalProgram, Solve, MosekSolver
from pydrake.trajectories import CompositeTrajectory
from pydrake.common import FindResourceOrThrow
from scipy.spatial import ConvexHull
import mcubes
from functools import partial
import matplotlib.pyplot as plt
from ipywidgets import widgets
import quadprog

from pathlib import Path
import sys

# add the parent directory of this notebook to the import path
parent = Path.cwd().parent
if str(parent) not in sys.path:
    sys.path.insert(0, str(parent))

from ciris_plant_visualizer import CIrisPlantVisualizer

In [31]:
# Cell 2: Set up the plant and scene graph, and initialize the CIrisPlantVisualizer
# builder = DiagramBuilder()
# plant, scene_graph = AddMultibodyPlantSceneGraph(builder, time_step=0.0)
# parser = Parser(plant, scene_graph)
# parser.SetAutoRenaming(True)

print("Setting up the plant and scene graph...")

# Replace DiagramBuilder with RobotDiagramBuilder
builder = RobotDiagramBuilder(time_step=0.0)
plant = builder.plant()
scene_graph = builder.scene_graph()
parser = Parser(plant, scene_graph)
parser.SetAutoRenaming(True)

Setting up the plant and scene graph...


In [32]:
# Add the robot
# gripper = parser.AddModels(file_name="../my_sdfs/wsg_2dof.sdf")[0]

print("Loading Panda robot models...")

# --- Add the Panda arm + hand ---
panda_arm  = parser.AddModels(url="package://drake_models/franka_description/urdf/panda_arm.urdf")[0]
# panda_hand = parser.AddModels(url="package://drake_models/franka_description/urdf/panda_hand.urdf")[0]
panda_hand = parser.AddModels(url="file:///home/julialopezgomez/optimisation-based-manipulation-planner/my_sdfs/panda_hand.urdf")[0]

print("Welding Panda arm and hand...")

# Weld arm base to world (identity)
plant.WeldFrames(
    plant.world_frame(),
    plant.GetFrameByName("panda_link0", panda_arm),
    RigidTransform())

# Weld hand to arm flange with X_PC: translation [0,0,0], RPY deg [0,0,-45]
X_8H = RigidTransform(RollPitchYaw(0.0, 0.0, -np.deg2rad(45.0)), [0.0, 0.0, 0.0])
plant.WeldFrames(
    plant.GetFrameByName("panda_link8", panda_arm),        # parent (P)
    plant.GetFrameByName("panda_hand", panda_hand),    # child  (C)
    X_8H)

print("Setting default finger joint positions...")

# Optional: set the default finger opening (each finger is a prismatic joint).
# 0.02 m on each finger → ~0.04 m total width. Adjust to taste.
# for j in ["panda_finger_joint1", "panda_finger_joint2"]:
#     plant.GetJointByName(j, panda_hand).set_default_translation(0.02)
    
 
    

   
# print("Welding arm joints to the world to lock them in place...")

# # Weld all arm joints to the world to lock them in place (we only want to move the hand/fingers in this example)
# locked_arm_joint_names = [f"panda_joint{i}" for i in range(1, 7)]

# default_grasp = np.array([
#     -1.28718907e+00,
#      1.31290357e+00,
#      1.18527863e+00,
#     -2.21273568e+00,
#     -1.58448521e+00,
#      2.03056033e+00,
#      1.66303822e+00,
#     -2.40000000e-02,
#      2.40000000e-02,
#      8.88178420e-16,
# ])

# all_joint_names = [
#     "panda_joint1",
#     "panda_joint2",
#     "panda_joint3",
#     "panda_joint4",
#     "panda_joint5",
#     "panda_joint6",
#     "panda_joint7",
#     "panda_finger_joint1",
#     "panda_finger_joint2",
#     "cap_to_base",
# ]
# q_by_name = dict(zip(all_joint_names, default_grasp))

# # weld all joints in locked_arm_joint_names to their default positions
# for jn in locked_arm_joint_names:
#     joint = plant.GetJointByName(jn, panda_arm)
#     joint.set_default_positions(np.array([q_by_name[jn]]))
#     joint.Lock(visualizer.task_space_diagram_context)
#     print(f"Welded joint {jn} to default position {q_by_name[jn]}. Locked: {joint.is_locked(visualizer.task_space_diagram_context)}")
    


Loading Panda robot models...
Welding Panda arm and hand...
Setting default finger joint positions...


In [33]:
#print all panda arm frame names



In [34]:
print("Adding the bottle cap and obstacles...")

cap = parser.AddModels(file_name="../my_sdfs/bottle_cap.sdf")[0]
# obstacle1 = parser.AddModels("../my_sdfs/obstacle.sdf")[0]
# # obstacle2 = parser.AddModels("my_sdfs/obstacle.sdf")[0]
# obstacle3 = parser.AddModels("../my_sdfs/obstacle.sdf")[0]

# Set welds
plant.WeldFrames(
    plant.world_frame(), 
    plant.GetFrameByName("base_link", cap),
    RigidTransform(RotationMatrix(), [0.5, 0, 0]))

# # Weld the obstacle to the world frame (adjust pose as needed)
# obstacle_pose1 = RigidTransform(RotationMatrix(), [0.51, 0.035, 0.02])  # Adjust position
# plant.WeldFrames(
#     plant.world_frame(),
#     plant.GetFrameByName("obstacle_link", obstacle1),
#     obstacle_pose1)

# # obstacle_pose2 = RigidTransform(RotationMatrix(), [-0.025, 0.05, 0.02])  # Adjust position
# # plant.WeldFrames(
# #     plant.world_frame(),
# #     plant.GetFrameByName("obstacle_link", obstacle2),
# #     obstacle_pose2)

# obstacle_pose3 = RigidTransform(RotationMatrix(), [0.465, -0.005, 0.02])  # Adjust position
# plant.WeldFrames(
#     plant.world_frame(),
#     plant.GetFrameByName("obstacle_link", obstacle3),
#     obstacle_pose3)

Adding the bottle cap and obstacles...


<WeldJoint name='world_welds_to_base_link' index=14 model_instance=4>

In [35]:
print("Finalising plant...")
print("Bodies:", plant.num_bodies(), "Joints:", plant.num_joints())


plant.Finalize()

inspector = scene_graph.model_inspector()
num_frames = len(list(inspector.GetAllFrameIds()))
num_geoms = sum(inspector.NumGeometriesForFrame(fid) for fid in inspector.GetAllFrameIds())
print("Frames:", num_frames, "Geometries:", num_geoms)


print("Plant finalised.")

print("Number of positions: ", plant.num_positions())

# Cell 3: Initialize the CIrisPlantVisualizer
q_star = np.zeros(plant.num_positions())


print("Initialising CspaceFreePolytope...")

# The object we will use to perform our certification
cspace_free_polytope = CspaceFreePolytope(
    plant, 
    scene_graph,
    SeparatingPlaneOrder.kAffine,
    q_star)

print("Initializing CIrisPlantVisualizer...")

visualizer = CIrisPlantVisualizer(
    plant,
    builder,
    scene_graph,
    cspace_free_polytope,
    viz_role=Role.kIllustration,
    allow_plus_3dof=True
)

print("Setting up the visualizer...")

visualizer.task_space_diagram.ForcedPublish(visualizer.task_space_diagram_context)


INFO:drake:Meshcat listening for connections at http://localhost:7003


Finalising plant...
Bodies: 16 Joints: 15
Frames: 16 Geometries: 90
Plant finalised.
Number of positions:  9
Initialising CspaceFreePolytope...
Initializing CIrisPlantVisualizer...
Visualisations won't work properly. Can't visualize the TC-Space of plants with more than 3-DOF. The first 3 DOF from the plant will be visualized
Setting up the visualizer...


In [36]:
sliders = []

plant_context = visualizer.plant_context
diagram = visualizer.task_space_diagram
diagram_context = visualizer.task_space_diagram_context

for i in range(plant.num_positions()):
    q_low = plant.GetPositionLowerLimits()[i]
    q_high = plant.GetPositionUpperLimits()[i]
    step = (q_high - q_low) / 100
    sliders.append(widgets.FloatSlider(
        min=q_low, max=q_high, 
        value=0, step=step, 
        description=f"q{i}"))
    
q = np.zeros(plant.num_positions())
def handle_slider_change(change, idx):
    q[idx] = change['new']
    print(visualizer.check_collision_q_by_ik(q))
    plant.SetPositions(plant_context, q)
    diagram.ForcedPublish(diagram_context)
    
idx = 0
for slider in sliders:
    slider.observe(partial(handle_slider_change, idx = idx), names='value')
    idx+=1

for slider in sliders:
    display(slider)

FloatSlider(value=0.0, description='q0', max=2.8973, min=-2.8973, step=0.057946)

FloatSlider(value=0.0, description='q1', max=1.7628, min=-1.7628, step=0.035255999999999996)

FloatSlider(value=0.0, description='q2', max=2.8973, min=-2.8973, step=0.057946)

FloatSlider(value=-0.0698, description='q3', max=-0.0698, min=-3.0718, step=0.03002)

FloatSlider(value=0.0, description='q4', max=2.8973, min=-2.8973, step=0.057946)

FloatSlider(value=0.0, description='q5', max=3.7525, min=-0.0175, step=0.0377)

FloatSlider(value=0.0, description='q6', max=2.8973, min=-2.8973, step=0.057946)

FloatSlider(value=0.0, description='q7', max=0.0, min=-0.045, step=0.00045)

FloatSlider(value=0.0, description='q8', max=3.14, min=-3.14, step=0.06280000000000001)

In [ ]:
# # Keep only the gripper rotation, fingers, and cap rotation free.
# # The arm joints 1-6 will be locked later at the grasp configuration.
# free_joint_names = [
#     "panda_joint7",
#     "panda_finger_joint1",
#     "panda_finger_joint2",
#     "cap_to_base",
# ]
# locked_arm_joint_names = [f"panda_joint{i}" for i in range(1, 7)]


# def lock_arm_to_lower_dim_manifold(q_nominal):
#     plant.SetPositions(plant_context, q_nominal)

#     for joint_name in locked_arm_joint_names:
#         plant.GetJointByName(joint_name, panda_arm).Lock(plant_context)

#     diagram.ForcedPublish(diagram_context)
#     print("Locked arm joints:", locked_arm_joint_names)
#     print("Free manifold joints:", free_joint_names)


## Get Grasping Configuration Through IK

In [ ]:
import numpy as np
from pydrake.math import RigidTransform, RotationMatrix
from pydrake.multibody.inverse_kinematics import InverseKinematics
from pydrake.solvers import Solve, SnoptSolver, IpoptSolver, OsqpSolver, ScsSolver

def solve_ik_place_frame_at_pose(
    # visualizer,
    plant,
    plant_context,
    frame_B,                                # the frame you want to place (e.g., panda_hand)
    X_WG: RigidTransform,                   # desired pose of frame_B in world
    pos_tol=1e-3,                           # box half-width [m]
    theta_tol=np.deg2rad(2),                # angular tolerance [rad]
    min_distance=None,                      # e.g., 1e-3 to keep clearance; None to ignore
    q_seed=None,                            # seed configuration
):
    
    ik = InverseKinematics(plant, plant_context)
    q = ik.q()

    # Position: keep the origin of frame_B in a small box around X_WG.translation()
    p_BQ_B = np.zeros(3)
    p = X_WG.translation()
    ik.AddPositionConstraint(
        frameB=frame_B, p_BQ=p_BQ_B,
        frameA=plant.world_frame(),
        p_AQ_lower=p - pos_tol, p_AQ_upper=p + pos_tol
    )

    # Orientation: bound frame_B’s rotation to the desired world rotation
    # This constrains angle between R_WB and X_WG.rotation() to be <= theta_tol.    
    ik.AddOrientationConstraint(
    frameAbar=plant.world_frame(),            # Ā
    R_AbarA=X_WG.rotation(),                  # desired world rotation
    frameBbar=frame_B,                        # B̄ = your hand frame
    R_BbarB=RotationMatrix(),                 # identity in B
    theta_bound=theta_tol                     # radians
)


    # Optional collision clearance (uses Proximity role; can be conservative for grasps)
    if min_distance is not None:
        ik.AddMinimumDistanceLowerBoundConstraint(min_distance)

    prog = ik.prog()
    if q_seed is None:
        q_seed = plant.GetPositions(plant_context)
    prog.SetInitialGuess(q, q_seed)

    # Pick an available solver. Orientation makes this nonlinear; SNOPT/IPOPT are best.
    result = Solve(prog)
    if not result.is_success():
        # try a second pass with a looser theta or different seed before giving up
        return None, result

    q_sol = result.GetSolution(q)

    return q_sol, result


In [ ]:
# Build diagram/contexts as you already do
plant_context = visualizer.plant_context
diagram = visualizer.task_space_diagram
diagram_context = visualizer.task_space_diagram_context

# Frames and goal pose
E   = plant.GetFrameByName("panda_hand", panda_hand)   # tool frame
Cap = plant.GetFrameByName("base_link", cap)           # cap frame

# World poses
X_WCap = plant.CalcRelativeTransform(plant_context, plant.world_frame(), Cap)

# Choose a grasp goal: 65 mm above cap origin (same orientation as Cap here)
# Rotate by 180 deg upside down
R_CapGoal = RotationMatrix(RollPitchYaw(np.pi, 0, 0))
X_CapGoal = RigidTransform(R_CapGoal, [0.0, 0.0, 0.105])
X_WG = X_WCap.multiply(X_CapGoal)  # desired world pose for the hand frame

# (Optional) lock fingers during IK
# for jn in ["panda_finger_joint1", "panda_finger_joint2"]:
#     plant.GetJointByName(jn, panda_hand).Lock(plant_context)


# Solve IK (function from previous message)
q_sol, res = solve_ik_place_frame_at_pose(
    # visualizer=visualizer,
    plant=plant,
    plant_context=plant_context,
    frame_B=E,
    X_WG=X_WG,
    pos_tol=0.0,
    theta_tol=np.deg2rad(0),
    min_distance=None,
    q_seed=plant.GetPositions(plant_context)
)

if q_sol is None:
    print("IK failed:", res.get_solver_id().name())
else:
    plant.SetPositions(plant_context, q_sol)
    diagram.ForcedPublish(diagram_context)
    print("IK succeeded. New q:", q_sol)


In [ ]:
# q: [-1.28718907e+00  1.31290357e+00  1.18527863e+00 -2.21273568e+00
#  -1.58448521e+00  2.03056033e+00  1.66303822e+00 -2.40000000e-02
#   2.40000000e-02  8.88178420e-16]

q_grasp = np.array([-1.28718907e+00,  
                    1.31290357e+00,  
                    1.18527863e+00, 
                    -2.21273568e+00,
                    -1.58448521e+00,  
                    2.03056033e+00,  
                    1.66303822e+00,
                    -2.40000000e-02,
                    2.40000000e-02,
                    8.88178420e-16])
plant.SetPositions(plant_context, q_grasp)
diagram.ForcedPublish(diagram_context)

## Grasping Configuration checker

Hand frame must be located at a certain height range from the cap (and rotation, so a transform), fingers at an offset (also a range) and allow full rotation of gripper hand and cap

In [ ]:
# Get panda_hand frame between 0.0105 m and 0.11 m above the cap base_link frame
# Fingers should be open between 0.024 and 0.025 m and -0.024 and -0.025 m
# Get a function that checks whether these constraints are satisfied

def check_grasp_constraints(q):
    old_q = plant.GetPositions(plant_context)
    plant.SetPositions(plant_context, q)
    # diagram.ForcedPublish(diagram_context)
    
    # Get the current pose of the hand frame
    X_WE = plant.CalcRelativeTransform(plant_context, plant.world_frame(), E)
    X_WCap = plant.CalcRelativeTransform(plant_context, plant.world_frame(), Cap)
    
    # Compute the relative transform from Cap to E
    X_CapE = X_WCap.inverse().multiply(X_WE)
    
    z_height = X_CapE.translation()[2]
    
    right_finger_joint = plant.GetJointByName("panda_finger_joint1", panda_hand)
    left_finger_joint  = plant.GetJointByName("panda_finger_joint2", panda_hand)
    
    right_finger_pos = right_finger_joint.get_translation(plant_context)
    left_finger_pos  = left_finger_joint.get_translation(plant_context)
    
    # Check height constraint
    height_ok = 0.0105 <= z_height <= 0.11
    
    # Check finger opening constraints
    fingers_ok = (-0.025 <= right_finger_pos <= -0.024) and (0.024 <= left_finger_pos <= 0.025)
    
    # Restore old q
    plant.SetPositions(plant_context, old_q)
    diagram.ForcedPublish(diagram_context)
    
    return height_ok and fingers_ok, z_height, right_finger_pos, left_finger_pos

In [ ]:
print("Checking grasp constraints for q_grasp:", check_grasp_constraints(q_grasp))

# Generate C-free for full franka arm with IRIS-ZO

In [ ]:
builder = visualizer.builder
plant = visualizer.plant
scene_graph = visualizer.scene_graph
q_star = visualizer.q_star
rat_fk = visualizer.rat_forward_kin
inspector = visualizer.model_inspector
diagram = visualizer.task_space_diagram
context = visualizer.task_space_diagram_context
cspace_free_polytope = visualizer.cspace_free_polytope

In [ ]:
from pydrake.all import IrisZoOptions, IrisNp2Options, CommonSampledIrisOptions, IrisFromCliqueCoverOptions, IrisInConfigurationSpaceFromCliqueCover
import logging

# from pydrake.all import set_log_level
# set_log_level("warn")
# logging.getLogger("pydrake").setLevel(logging.WARNING)


In [ ]:

model_instances = [plant.GetModelInstanceByName("panda"), plant.GetModelInstanceByName("panda_hand"), plant.GetModelInstanceByName("bottle_cap")]

generator = RandomGenerator(1234)
checker = SceneGraphCollisionChecker(
    model=diagram,
    robot_model_instances=model_instances,
    edge_step_size=0.01,
)


common_sampled_iris_options = CommonSampledIrisOptions()
# common_sampled_iris_options.num_particles = 1000
# common_sampled_iris_options.tau = 0.5
common_sampled_iris_options.delta = 0.05
common_sampled_iris_options.epsilon = 0.01
common_sampled_iris_options.max_iterations = 10
# common_sampled_iris_options.max_iterations_separating_planes = 20
# common_sampled_iris_options.max_separating_planes_per_iteration = 10
parallelism = Parallelism(32)
common_sampled_iris_options.parallelism = parallelism
common_sampled_iris_options.verbose = True
# common_sampled_iris_options.require_sample_point_is_contained = True
common_sampled_iris_options.configuration_space_margin = 0.00001
# common_sampled_iris_options.relax_margin = True
common_sampled_iris_options.termination_threshold = 1e-5
common_sampled_iris_options.relative_termination_threshold = 1e-4
common_sampled_iris_options.remove_all_collisions_possible = False
# common_sampled_iris_options.random_seed = 1234
# common_sampled_iris_options.mixing_steps = 50
# common_sampled_iris_options.sample_particles_in_parallel = False


iris_zo_options = IrisZoOptions()
iris_zo_options.bisection_steps=30
iris_zo_options.sampled_iris_options=common_sampled_iris_options

# iris_np2_options = IrisNp2Options()
# iris_np2_options.sampled_iris_options = common_sampled_iris_options

options = IrisFromCliqueCoverOptions()
options.num_points_per_visibility_round = 200
options.coverage_termination_threshold = 0.9
options.parallelism = parallelism
# options.iris_options.configuration_space_margin = 0.0001
options.iris_options = iris_zo_options
# options.iris_options = iris_np2_options
# See https://github.com/RobotLocomotion/drake/issues/21343
regions = IrisInConfigurationSpaceFromCliqueCover(checker, options, generator, [])

print("\nnumber of regions: ", len(regions))


# visualizer.visualize_collision_constraint(factor=1, num_points=30, filled_polytopes=regions)

In [ ]:
regions


In [ ]:
for i, region in enumerate(regions):
    print(f"Region {i}:")
    # print("  planes:", region.A())
    print("  Shape:", region.A().shape)
    # print("  b:", region.b())
    print("  Shape:", region.b().shape)

In [ ]:

print("Plant num positions:", plant.num_positions())
print("Position names:")
for i, name in enumerate(plant.GetPositionNames()):
    print(i, name)

In [ ]:
def joint_position_index(plant, joint_name, model_instance=None):
    if model_instance is None:
        joint = plant.GetJointByName(joint_name)
    else:
        joint = plant.GetJointByName(joint_name, model_instance)

    if joint.num_positions() != 1:
        raise ValueError(
            f"Joint {joint_name} has {joint.num_positions()} positions; "
            "this helper assumes a 1-DOF joint."
        )

    return joint.position_start()


active_indices = [
    joint_position_index(plant, "panda_joint7", panda_arm),
    joint_position_index(plant, "panda_finger_joint1", panda_hand),
    joint_position_index(plant, "panda_finger_joint2", panda_hand),
    joint_position_index(plant, "cap_to_base", cap)
]

active_indices = list(active_indices)
fixed_indices = [i for i in range(plant.num_positions()) if i not in active_indices]

print("Active indices:", active_indices)
print("Fixed indices:", fixed_indices)

for i, name in enumerate(plant.GetPositionNames()):
    tag = "ACTIVE" if i in active_indices else "fixed"
    print(f"{i:2d}: {name:40s} {tag}")

In [ ]:
assert q_grasp.shape[0] == plant.num_positions()

def make_full_dimensional_slice_box(
    plant,
    q_fixed,
    active_indices,
    fixed_half_width=1e-4, 
    default_revolute_bound=np.pi,
):
    nq = plant.num_positions()

    lb = plant.GetPositionLowerLimits().copy()
    ub = plant.GetPositionUpperLimits().copy()

    active_set = set(active_indices)

    for i in range(nq):
        if i not in active_set:
            lb[i] = q_fixed[i] - fixed_half_width
            ub[i] = q_fixed[i] + fixed_half_width

    # Safety check.
    if np.any(lb >= ub):
        bad = np.where(lb >= ub)[0]
        raise ValueError(f"Invalid bounds at indices {bad}: lb >= ub")

    return HPolyhedron.MakeBox(lb, ub), lb, ub


slice_box, slice_lb, slice_ub = make_full_dimensional_slice_box(
    plant=plant,
    q_fixed=q_grasp,
    active_indices=active_indices,
    fixed_half_width=1e-4,   # try 1e-5 later if this works
)

### Run Clique-Cover IRIS inside that slice/bounding region



In [ ]:
from pydrake.geometry.optimization import IrisOptions

iris_options = IrisOptions()
iris_options.bounding_region = slice_box
iris_options.require_sample_point_is_contained = True

# Since this is called from clique cover, Drake recommends using only one
# internal IRIS iteration to avoid discarding clique information.
iris_options.iteration_limit = 1

iris_options.configuration_space_margin = 1e-5
iris_options.termination_threshold = -1
iris_options.relative_termination_threshold = -1
iris_options.random_seed = 1234
iris_options.mixing_steps = 50

options = IrisFromCliqueCoverOptions()
options.iris_options = iris_options
options.coverage_termination_threshold = 0.99
options.num_points_per_visibility_round = 500
options.num_points_per_coverage_check = 2000
options.parallelism = Parallelism(num_threads=8)

regions_4d_slice_10d = IrisInConfigurationSpaceFromCliqueCover(
    checker,
    options,
    generator,
    [],
)

print("Number of 10D slice regions:", len(regions_4d_slice_10d))
print("Ambient dimension:", regions_4d_slice_10d[0].ambient_dimension())

## Walk around c-free polytope

In [ ]:
def walk_around_cfree_regions(q_start, regions, num_steps=10):
    """
    Walk around the c-free regions starting from q_start.
    For each step, randomly select a region and sample a point within it.
    """
    current_q = q_start.copy()
    path = [current_q.copy()]

    for step in range(num_steps):
        # Randomly select a region
        region_idx = np.random.randint(len(regions))
        region = regions[region_idx]

        # Sample a point within the selected region
        A = region.A()
        b = region.b()

        # Use quadprog to sample a point within the polytope defined by Ax <= b
        # We will solve a quadratic program to find a feasible point
        n_vars = A.shape[1]
        P = np.eye(n_vars)  # Identity matrix for the quadratic term
        q = np.zeros(n_vars)  # Zero vector for the linear term

        # Solve the quadratic program: minimize (1/2)x^T P x + q^T x subject to Ax <= b
        try:
            sol = quadprog.solve_qp(P, q, A.T, b)[0]
            current_q = sol
            path.append(current_q.copy())
            print(f"Step {step + 1}: Moved to region {region_idx}, new q: {current_q}")
        except Exception as e:
            print(f"Step {step + 1}: Failed to sample from region {region_idx}, error: {e}")

    return path



def is_q_within_any_region(q, regions):
    for region in regions:
        if np.all(region.A() @ q <= region.b()):
            return True
    return False


def find_any_q_within_regions(regions):
    for region in regions:
        # Sample a point within the region using quadprog
        A = region.A()
        b = region.b()
        n_vars = A.shape[1]
        P = np.eye(n_vars)  # Identity matrix for the quadratic term
        q = np.zeros(n_vars)  # Zero vector for the linear term

        try:
            sol = quadprog.solve_qp(P, q, A.T, b)[0]
            return sol
        except Exception as e:
            print(f"Failed to sample from region, error: {e}")
    return None

q_start = find_any_q_within_regions(regions)




print("Starting walk around c-free regions from q_start:", q_start)
assert is_q_within_any_region(q_start, regions), "q_start is not within any c-free region!"

walk_path = walk_around_cfree_regions(q_start=q_start, regions=regions, num_steps=10)



# Trajectory Generation

### Traj to grasp pose

Traverse the c-free regions until the feasible grasping configuration is reached. Then generate trajectory from current configuration to goal configuration.